In [1]:
from pathlib import Path

from vectormesh.data.cache import VectorCache

artefacts = Path("../artefacts")
trainpath = next(artefacts.glob("*bert*train/"))
validpath = next(artefacts.glob("*bert*valid/"))
traincache = VectorCache.load(path=trainpath)
validcache = VectorCache.load(path=validpath)


2026-06-06 10:48:41.857 | SUCCESS  | vectormesh.data.cache:load:140 - Cache loaded from ../artefacts/20260113075724_aktes_theshold_50_d97342_legal_bert_train
2026-06-06 10:48:41.862 | SUCCESS  | vectormesh.data.cache:load:140 - Cache loaded from ../artefacts/20260113080216_aktes_theshold_50_d97342_legal_bert_valid


We load all data

In [2]:
from vectormesh.data import OneHot

onehot = OneHot(num_classes=32, label_col="labels", target_col="onehot")
train_oh = traincache.dataset.map(onehot)
valid_oh = validcache.dataset.map(onehot)

Set the output label to a onehot encoded vector, and add fixed padding to the dataloader.

In [3]:
from torch.utils.data import DataLoader

from vectormesh.components import FixedPadding
from vectormesh.data import Collate

collate_fn = Collate(
    embedding_col="legal_dutch",
    target_col="onehot",
    padder=FixedPadding(max_chunks=30),
)

trainloader = DataLoader(train_oh, batch_size=32, shuffle=True, collate_fn=collate_fn)
validloader = DataLoader(valid_oh, batch_size=32, shuffle=False, collate_fn=collate_fn)

A Mixture-of-Experts layer replaces a single dense feed-forward block with many independent "expert" sub-networks plus a small trainable gating/router network. See the paper `Outrageously Large Neural Networks` in the `references/` folder for more details.

In [4]:
from vectormesh.components import MaskedMeanAggregator, NeuralNet, Serial
from vectormesh.components.gating import MoE

moe = MoE(
    experts=[
        NeuralNet(hidden_size=768, out_size=32),
        NeuralNet(hidden_size=768, out_size=32),
        NeuralNet(hidden_size=768, out_size=32),
        NeuralNet(hidden_size=768, out_size=32),
    ],
    hidden_size=768,
    out_size=32,
)

pipeline = Serial([MaskedMeanAggregator(), moe])

Obviously, this can be improved (see notebook `2_design` for inspiration) but that is left as an exercise for the reader.

In [5]:
import torch
import torch.optim as optim
from mltrainer import ReportTypes, Trainer, TrainerSettings

from vectormesh.components.metrics import F1Score
from vectormesh.data.vectorizers import detect_device

device = detect_device()
print(f"Using device: {device}")

log_dir = Path("tmp").absolute()

settings = TrainerSettings(
    epochs=3,
    metrics=[F1Score()],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(trainloader),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

loss_fn = torch.nn.BCEWithLogitsLoss()

trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=trainloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

2026-06-06 10:48:43.314 | INFO     | mltrainer.settings:check_path:60 - Created logdir /Users/rgrouls/code/courses/vectormesh/notebooks/tmp
2026-06-06 10:48:43.314 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to /Users/rgrouls/code/courses/vectormesh/notebooks/tmp/20260606-104843
2026-06-06 10:48:43.326 | INFO     | mltrainer.trainer:__init__:66 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


Using device: mps


In [6]:
trainer.loop()

100%|██████████| 486/486 [00:03<00:00, 130.88it/s]
2026-06-06 10:48:49.679 | INFO     | mltrainer.trainer:report:215 - Epoch 0 train 0.0704 test 0.0493 metric ['0.7838']
100%|██████████| 486/486 [00:03<00:00, 136.92it/s]
2026-06-06 10:48:55.984 | INFO     | mltrainer.trainer:report:215 - Epoch 1 train 0.0446 test 0.0418 metric ['0.8148']
100%|██████████| 486/486 [00:03<00:00, 144.62it/s]
2026-06-06 10:49:01.966 | INFO     | mltrainer.trainer:report:215 - Epoch 2 train 0.0409 test 0.0391 metric ['0.8176']
100%|██████████| 3/3 [00:18<00:00,  6.21s/it]


In [7]:
import shutil

shutil.rmtree("tmp/", ignore_errors=True)
shutil.rmtree("logs/", ignore_errors=True)
shutil.rmtree("demo/", ignore_errors=True)